In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict
import os

# 1. 加载环境变量，初始化大模型
load_dotenv()
llm = ChatOpenAI(
    model="qwen-plus",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    timeout=30,
)

# 2. 定义状态
class State(TypedDict):
    question: str
    llm_result: str
    num1: int
    num2: int
    python_result: int
    check_result: str
    retry_count: int

# 节点1：LLM直接计算结果
def llm_calculate_node(state: State) -> dict:
    retry_count = state.get("retry_count", 0)
    print(f"\n=== 节点1: LLM计算 (第{retry_count}次尝试) ===")
    question = state["question"]
    result = llm.invoke(f"请直接计算这个数学题，只输出最终答案的数字：{question}")
    llm_result = result.content.strip()
    print(f"LLM计算结果: {llm_result}")
    return {"llm_result": llm_result}

# 节点2：LLM提取两个数字
def extract_numbers_node(state: State) -> dict:
    retry_count = state.get("retry_count", 0)
    print(f"\n=== 节点2: LLM提取数字 (第{retry_count}次尝试) ===")
    question = state["question"]
    extract_prompt = f"从以下问题中提取两个数字，格式输出为：数字1|数字2（只输出这两个数字，不要其他内容）\\n问题：{question}"
    extract_result = llm.invoke(extract_prompt)
    parts = extract_result.content.strip().split("|")
    num1 = int(parts[0].strip())
    num2 = int(parts[1].strip())
    print(f"提取数字：num1={num1}, num2={num2}")
    return {"num1": num1, "num2": num2}

# 节点3：Python函数计算结果（可靠）
def python_calculate_node(state: State) -> dict:
    retry_count = state.get("retry_count", 0)
    print(f"\n=== 节点3: Python计算 (第{retry_count}次尝试) ===")
    num1 = state["num1"]
    num2 = state["num2"]
    python_result = num1 + num2  # 正确计算
    print(f"Python计算结果: {num1} + {num2} = {python_result}")
    return {"python_result": python_result}

# 节点4：检查结果是否一致
def check_node(state: State) -> dict:
    retry_count = state.get("retry_count", 0)
    print(f"\n=== 节点4: 检查结果 (第{retry_count}次尝试) ===")
    import re
    llm_result = state["llm_result"]
    python_result = state["python_result"]
    print(f"LLM结果: {llm_result}, Python结果: {python_result}")
    llm_num_str = re.sub(r'[^0-9-]', '', str(llm_result))
    try:
        llm_num = int(llm_num_str)
        if llm_num == python_result:
            check_result = "匹配"
            print("检查结果：匹配")
        else:
            check_result = "不匹配"
            print(f"检查结果：不匹配 (LLM: {llm_num}, Python: {python_result})")
    except ValueError:
        check_result = "不匹配"
        print(f"检查结果：不匹配 (无法解析LLM结果)")
    return {"check_result": check_result}

# 节点5：递增重试计数器
def increment_retry_node(state: State) -> dict:
    current_count = state.get("retry_count", 0) + 1
    print(f"\n=== 重试计数器: {current_count} ===")
    return {"retry_count": current_count}

# 创建图
graph = StateGraph(State)

# 添加节点
graph.add_node("llm_calculate", llm_calculate_node)
graph.add_node("extract_numbers", extract_numbers_node)
graph.add_node("python_calculate", python_calculate_node)
graph.add_node("check", check_node)
graph.add_node("increment_retry", increment_retry_node)

# 定义边
graph.add_edge("llm_calculate", "extract_numbers")
graph.add_edge("extract_numbers", "python_calculate")
graph.add_edge("python_calculate", "check")
graph.add_edge("check", "increment_retry")

# 条件分支
def route_based_on_check(state: State) -> str:
    retry_count = state.get("retry_count", 0)
    print(f"\n--- 路由决策: {state['check_result']}, 次数={retry_count} ---")
    if retry_count >= 5:
        print("达到最大重试次数")
        return "匹配"
    if state["check_result"] == "匹配":
        return "匹配"
    return "不匹配"

graph.add_conditional_edges(
    "increment_retry",
    route_based_on_check,
    {
        "匹配": END,
        "不匹配": "llm_calculate"
    }
)

# 入口
graph.set_entry_point("llm_calculate")

# 编译（简单场景不需要checkpointer）
app = graph.compile()

# 运行
initial_state = {
    "question": "计算 123456 + 789012 等于多少？",
    "retry_count": 0
}
print("=" * 50)
print("开始执行...")
result = app.invoke(initial_state)

# 打印最终结果
print("\n" + "=" * 50)
print("最终结果：")
print(f"问题: {result['question']}")
print(f"LLM计算结果: {result['llm_result']}")
print(f"提取的数字: {result['num1']} + {result['num2']}")
print(f"Python计算结果: {result['python_result']}")
print(f"最终检查: {result['check_result']}")
print(f"总重试次数: {result.get('retry_count', 0)}")